In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [ ]:
df.head()
df['State'].value_counts()

In [ ]:
df_tx= df[df['State'] == 'TX']

In [ ]:
df_tx.shape

In [ ]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [ ]:

df_tx=df_tx[columns]

In [ ]:
df_tx.columns

In [ ]:
df_tx.notnull().sum()

In [ ]:
df_tx.fillna({
    'Temperature(F)': df_tx['Temperature(F)'].mean(),
    'Humidity(%)': df_tx['Humidity(%)'].mean(),
    'Pressure(in)': df_tx['Pressure(in)'].mean(),
    'Visibility(mi)': df_tx['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_tx['Wind_Speed(mph)'].mean(),
    'Precipitation(in)':df_tx['Precipitation(in)'].mean(),
    'Wind_Chill(F)': df_tx['Wind_Chill(F)'].mean()
},inplace=True)

In [ ]:
df_tx.fillna({
    'Weather_Condition': df_tx['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_tx['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [ ]:
df_tx['Start_Time'] = pd.to_datetime(df_tx['Start_Time'], errors='coerce')
df_tx['hour']= df_tx['Start_Time'].dt.hour
df_tx['day']= df_tx['Start_Time'].dt.dayofweek
df_tx['month']= df_tx['Start_Time'].dt.month

In [ ]:
df_tx.sample(12)
df_tx.notnull().sum()
df_tx.fillna({
    'hour': df_tx['hour'].mode()[0],
    'day': df_tx['day'].mode()[0],
    'month': df_tx['month'].mode()[0]
}, inplace=True)

In [ ]:
df_tx['is_rush_hour'] = df_tx['hour'].isin(range(7,10)).astype(int) | df_tx['hour'].isin(range(16,20)).astype(int)
df_tx['is_weekend'] = (df_tx['day'] >= 5).astype(int)
df_tx['is_winter'] = df_tx['month'].isin([11,12,1,2]).astype(int)

In [ ]:
df_tx.notnull().sum()
df_tx.drop(columns= ['Start_Time'], inplace=True)

In [ ]:

df_tx.notnull().sum()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [ ]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [ ]:
import numpy as np

df_tx = df_tx.copy()

# remove impossible coordinates
df_tx = df_tx[
    (df_tx['Start_Lat'].between(-90, 90)) &
    (df_tx['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_tx = df_tx[df_tx['Temperature(F)'] > -100]   # sanity check
df_tx = df_tx[df_tx['Wind_Speed(mph)'] >= 0]
df_tx = df_tx[df_tx['Pressure(in)'] > 0]
df_tx = df_tx[df_tx['Humidity(%)'].between(0, 100)]
df_tx = df_tx[df_tx['Visibility(mi)'] >= 0]
df_tx = df_tx[df_tx['Precipitation(in)'] >= 0]

df_tx = df_tx[
    (df_tx['Wind_Chill(F)'] > -100) &
    (df_tx['Wind_Chill(F)'] < 150)
]

In [ ]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)',
    'Wind_Chill(F)',
    'Precipitation(in)'
]

for col in numeric_cols:
    df_tx = iqr_filter(df_tx, col)

In [ ]:
lat_min, lat_max = df_tx['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_tx['Start_Lng'].quantile([0.01, 0.99])

df_tx = df_tx[
    (df_tx['Start_Lat'].between(lat_min, lat_max)) &
    (df_tx['Start_Lng'].between(lng_min, lng_max))
]

In [ ]:
x= df_tx.drop(columns=['Severity'])
y= df_tx['Severity']


In [ ]:
df_tx['Severity'].value_counts()

In [ ]:
y1=df_tx['Severity']-1

In [ ]:
y1.value_counts()

In [ ]:
y1 = (df_tx['Severity'] >= 3).astype(int)

In [ ]:
y1.value_counts()

In [ ]:
df_tx.shape

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
import joblib
joblib.dump(model2,"pa_model.pkl")